In [ ]:
"""
Configuration for multilingual fact-verification system
"""
import os

class Config:

    try:
        BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    except NameError:
        BASE_DIR = os.getcwd()
    DATA_DIR = os.path.join(BASE_DIR, 'data')
    CACHE_DIR = os.path.join(DATA_DIR, 'cache')
    INDEX_DIR = os.path.join(DATA_DIR, 'indices')
    MODEL_DIR = os.path.join(BASE_DIR, 'models')

    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(INDEX_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    SUPPORTED_LANGUAGES = ['ar', 'ta']
    LANGUAGE_NAMES = {
        'ar': 'Arabic',
        'ta': 'Tamil',
    }

    # Dataset settings
    FINEWIKI_SUBSET_SIZE = 50000
    FINEWEB2_SUBSET_SIZE = 80000
    XFACT_SIZE = None
    ENGLISH_SUBSET_SIZE  = 5000

    # Model settings
    EMBEDDING_MODEL = "sentence-transformers/LaBSE"
    EMBEDDING_DIM = 768
    NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

    # Stage 1: Embedding fine-tuning
    STAGE1_EPOCHS = 1
    STAGE1_BATCH_SIZE = 16
    STAGE1_LEARNING_RATE = 2e-5
    STAGE1_WARMUP_STEPS = 500
    STAGE1_NUM_PAIRS = 100000
    STAGE1_POSITIVE_RATIO = 0.5
    STAGE1_EVAL_STEPS = 1000
    STAGE1_SAVE_BEST = True
    STAGE1_VALIDATION_SPLIT = 0.1
    STAGE1_LOSS_TYPE = 'contrastive'

    # Stage 2: Threshold optimization ✅ renamed to match EvidenceGate
    STAGE2_NLI_RANGE        = [0.45, 0.50, 0.55, 0.60, 0.65]
    STAGE2_SIMILARITY_RANGE = [0.70, 0.75, 0.80, 0.85]
    STAGE2_SUPPORT_RANGE    = [0.30, 0.40, 0.50, 0.60]

    # Default thresholds ✅ renamed and recalibrated
    NLI_MIN        = 0.55
    SIMILARITY_MIN = 0.80
    SUPPORT_MIN    = 0.40

    # Retrieval settings
    TOP_K_RESULTS = 10
    FAISS_INDEX_TYPE = 'IndexFlatIP'

    # File paths
    EVIDENCE_INDEX_PATH       = os.path.join(INDEX_DIR, 'evidence_index.faiss')
    DOCUMENTS_PATH            = os.path.join(INDEX_DIR, 'documents.pkl')
    FINETUNED_MODEL_PATH      = os.path.join(MODEL_DIR, 'embeddings_finetuned')
    OPTIMIZED_THRESHOLDS_PATH = os.path.join(MODEL_DIR, 'optimized_thresholds.json')

    LOG_LEVEL = 'INFO'

    @classmethod
    def display(cls):
        print("=" * 60)
        print("CONFIGURATION")
        print("=" * 60)
        print(f"Languages       : {', '.join(cls.SUPPORTED_LANGUAGES)}")
        print(f"FineWiki subset : {cls.FINEWIKI_SUBSET_SIZE} articles/lang")
        print(f"FineWeb2 subset : {cls.FINEWEB2_SUBSET_SIZE} docs/lang")
        print(f"Training pairs  : {cls.STAGE1_NUM_PAIRS} (pos/neg: {cls.STAGE1_POSITIVE_RATIO:.0%}/{1-cls.STAGE1_POSITIVE_RATIO:.0%})")
        print(f"Embedding model : {cls.EMBEDDING_MODEL}")
        print(f"NLI model       : {cls.NLI_MODEL}")
        print(f"Epochs          : {cls.STAGE1_EPOCHS}")
        print(f"Batch size      : {cls.STAGE1_BATCH_SIZE}")
        print(f"Thresholds      : nli={cls.NLI_MIN}, similarity={cls.SIMILARITY_MIN}, support={cls.SUPPORT_MIN}")
        print("=" * 60)

In [ ]:
!pip install langdetect


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 37.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a0e383a0b81d89fc94c897f9c72229fe0c7892e5e25c6c9ad38a7598f34c4354
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
"""
Language detection and text preprocessing utilities
"""
from langdetect import detect, LangDetectException
import re

def detect_language(text: str) -> str:
    """
    Detect language of input text

    Args:
        text: Input text

    Returns:
        Language code ('ar', 'ta', or 'unknown')
    """
    try:
        lang = detect(text)
        # Map detected language to our supported languages
        if lang in ['ar', 'ta']:
            return lang
        return 'unknown'
    except LangDetectException:
        return 'unknown'

def preprocess_text(text: str, lang: str = None) -> str:
    """
    Preprocess text for embedding

    Args:
        text: Input text
        lang: Language code (optional)

    Returns:
        Cleaned text
    """
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove special characters but keep language-specific characters
    # Keep Urdu/Tamil scripts
    text = text.strip()

    # Truncate if too long (optional)
    max_length = 512
    if len(text) > max_length:
        text = text[:max_length]

    return text

def split_into_chunks(text: str, chunk_size: int = 200, overlap: int = 50) -> list:
    """
    Split text into overlapping chunks

    Args:
        text: Input text
        chunk_size: Size of each chunk
        overlap: Overlap between chunks

    Returns:
        List of text chunks
    """
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += (chunk_size - overlap)

    return chunks

def truncate_text(text: str, max_chars: int = 500) -> str:
    """
    Truncate text to maximum characters

    Args:
        text: Input text
        max_chars: Maximum characters

    Returns:
        Truncated text
    """
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."

In [ ]:
from datasets import load_dataset
import pickle
import os
from tqdm import tqdm

try:
    BASE_PATH = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    BASE_PATH = os.getcwd()

FINEWEB2_LANG_MAP = {
    "ar": "arb_Arab",
    "ta": "tam_Taml",
}

def download_finewiki(language: str, subset_size: int = None):
    print(f"\nDownloading FineWiki ({language})...")
    try:
        dataset = load_dataset(
            "HuggingFaceFW/finewiki",
            language,
            split="train",
            streaming=True  # ✅ prevents full download
        )
    except Exception as e:
        print(f"⚠️ Could not load FineWiki for {language}: {e}")
        return []

    articles = []
    for i, item in enumerate(tqdm(dataset, total=subset_size, desc=f"Processing {language}")):
        if subset_size and i >= subset_size:
            break
        text = item.get('text', '')
        if len(text) > 100:
            articles.append({
                'title': item.get('title', ''),
                'text': text,
                'language': language
            })

    print(f"✅ {len(articles)} articles for {language}")
    return articles


def download_fineweb2(language: str, subset_size: int = 30000):
    print(f"\nDownloading FineWeb2 ({language})...")
    config_name = FINEWEB2_LANG_MAP.get(language)
    if config_name is None:
        raise ValueError(f"Unsupported language: {language}")

    dataset = load_dataset(
        "HuggingFaceFW/fineweb-2",
        name=config_name,
        split="train",
        streaming=True
    )

    documents = []
    for i, item in enumerate(tqdm(dataset, total=subset_size, desc=f"Downloading {language}")):
        if i >= subset_size:
            break
        text = item.get('text', '')
        if len(text) > 100:
            documents.append({'text': text, 'language': language})

    print(f"✅ {len(documents)} documents for {language}")
    return documents


def download_xfact():
    print("\nDownloading XFACT...")
    dataset = load_dataset("utahnlp/x-fact", "all_languages")

    splits = {}
    for split_name in ['train', 'dev', 'test']:
        if split_name in dataset:
            examples = [
                ex for ex in dataset[split_name]
                if ex.get('language') in ['ar', 'ta']
            ]
            splits[split_name] = examples
            print(f"  {split_name}: {len(examples)} examples")

    print("✅ XFACT downloaded")
    return splits


def save_data(data, filename):
    filepath = os.path.join(Config.CACHE_DIR, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    print(f"💾 Saved to {filepath}")


def load_data(filename):
    filepath = os.path.join(Config.CACHE_DIR, filename)
    with open(filepath, 'rb') as f:
        return pickle.load(f)


def main():
    print("=" * 60)
    print("DOWNLOADING DATASETS")
    print("=" * 60)

    Config.display()

    # FineWiki
    finewiki_data = {}
    for lang in Config.SUPPORTED_LANGUAGES:
        finewiki_data[lang] = download_finewiki(lang, Config.FINEWIKI_SUBSET_SIZE)
    save_data(finewiki_data, 'finewiki_data.pkl')  # ✅ save after loop

    # FineWeb2
    fineweb2_data = {}
    for lang in Config.SUPPORTED_LANGUAGES:
        fineweb2_data[lang] = download_fineweb2(lang, Config.FINEWEB2_SUBSET_SIZE)
    save_data(fineweb2_data, 'fineweb2_data.pkl')  # ✅ save after loop

    # XFACT
    xfact_data = download_xfact()
    save_data(xfact_data, 'xfact_data.pkl')

    print("\n✅ ALL DATASETS DOWNLOADED")
    print(f"FineWiki : {sum(len(v) for v in finewiki_data.values())} articles")
    print(f"FineWeb2 : {sum(len(v) for v in fineweb2_data.values())} documents")
    print(f"XFACT    : {sum(len(v) for v in xfact_data.values())} examples")


if __name__ == "__main__":
    main()

DOWNLOADING DATASETS
CONFIGURATION
Languages       : ar, ta
FineWiki subset : 50000 articles/lang
FineWeb2 subset : 80000 docs/lang
Training pairs  : 100000 (pos/neg: 50%/50%)
Embedding model : sentence-transformers/LaBSE
NLI model       : MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Epochs          : 1
Batch size      : 16
Thresholds      : nli=0.55, similarity=0.8, support=0.4



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Processing ar: 100%|██████████| 50000/50000 [00:54<00:00, 914.20it/s] 


✅ 48209 articles for ar



...

Generating dev split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating ood split: 0 examples [00:00, ? examples/s]

Generating zeroshot split: 0 examples [00:00, ? examples/s]

  train: 2664 examples
  dev: 354 examples
  test: 535 examples
✅ XFACT downloaded
💾 Saved to /content/data/cache/xfact_data.pkl

✅ ALL DATASETS DOWNLOADED
FineWiki : 98145 articles
FineWeb2 : 160000 documents
XFACT    : 3553 examples


In [ ]:
dataset = load_dataset("utahnlp/x-fact", "all_languages")

langs = set(ex["language"] for ex in dataset["train"])
print(sorted(langs))


['ar', 'de', 'es', 'hi', 'id', 'it', 'ka', 'pl', 'pt', 'ro', 'sr', 'ta', 'tr']


In [ ]:
import json
import pickle
import os
from collections import Counter

def convert_label(raw_label: str) -> str:
    label = raw_label.lower().strip()
    if label == 'true':
        return 'SUPPORTS'
    elif label == 'false':
        return 'REFUTES'
    else:
        return 'NEI'

def build_jsonl(examples: list) -> list:
    output = []
    for ex in examples:
        evidence_parts = []
        for i in range(1, 6):
            ev = ex.get(f'evidence_{i}', '')
            if ev and ev != '<DUMMY_EVIDENCE>' and ev is not None:
                evidence_parts.append(ev.strip())

        evidence = ' '.join(evidence_parts[:2])

        output.append({
            'claim':    ex['claim'],
            'evidence': evidence,
            'label':    convert_label(ex['label']),
            'language': ex['language']
        })
    return output

# Load XFACT
with open('/content/data/cache/xfact_data.pkl', 'rb') as f:
    xfact_data = pickle.load(f)

os.makedirs('data/final', exist_ok=True)

split_map = {'train': 'train', 'dev': 'val', 'test': 'test'}

for xfact_split, file_split in split_map.items():
    examples = xfact_data.get(xfact_split, [])
    converted = build_jsonl(examples)

    path = f'data/final/{file_split}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    labels = Counter(ex['label'] for ex in converted)
    langs  = Counter(ex['language'] for ex in converted)
    print(f"{file_split}: {len(converted)} examples | labels: {dict(labels)} | langs: {dict(langs)}")

print("\n✅ data/final/ ready for training")


train: 2664 examples | labels: {'NEI': 1029, 'REFUTES': 1315, 'SUPPORTS': 320} | langs: {'ta': 1097, 'ar': 1567}
val: 354 examples | labels: {'REFUTES': 170, 'NEI': 135, 'SUPPORTS': 49} | langs: {'ar': 209, 'ta': 145}
test: 535 examples | labels: {'NEI': 213, 'REFUTES': 251, 'SUPPORTS': 71} | langs: {'ta': 221, 'ar': 314}

✅ data/final/ ready for training


In [ ]:
"""
NLI-based reliability scoring
Replaces FFT frequency analysis with actual entailment detection
"""
from transformers import pipeline
import numpy as np

class ReliabilityScorer:
    """
    Scores how much evidence supports a claim using NLI entailment.
    Works for Arabic, Tamil, and English (your three languages).
    """

    def __init__(self):
        # mDeBERTa works for Arabic, Tamil, English — unlike bart-large-mnli which is English only
        self.nli = pipeline(
            "zero-shot-classification",
            model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
            device=-1  # CPU, change to 0 if you have GPU
        )

    def compute_reliability(self, claim: str, evidence_text: str) -> float:
        """
        Returns probability that evidence supports the claim (0 to 1)

        Args:
            claim: The claim being verified
            evidence_text: A single piece of retrieved evidence

        Returns:
            Entailment probability as reliability score
        """
        result = self.nli(
            evidence_text[:512],  # truncate to avoid token limit
            candidate_labels=["supported", "refuted"],
            hypothesis_template="The following claim is {}: " + claim
        )

        supported_idx = result['labels'].index('supported')
        return float(result['scores'][supported_idx])

    def batch_compute_reliability(self, claim: str, evidence_texts: list) -> np.ndarray:
        """
        Score multiple evidence pieces against one claim

        Args:
            claim: The claim being verified
            evidence_texts: List of evidence strings

        Returns:
            Array of reliability scores
        """
        scores = []
        for text in evidence_texts:
            score = self.compute_reliability(claim, text)
            scores.append(score)
        return np.array(scores)

    def average_reliability(self, claim: str, evidence_texts: list) -> float:
        """
        Average NLI score across all evidence — use this as nli_score in EvidenceGate

        Args:
            claim: The claim being verified
            evidence_texts: List of evidence strings

        Returns:
            Mean reliability score (0 to 1)
        """
        scores = self.batch_compute_reliability(claim, evidence_texts)
        return float(np.mean(scores)) if len(scores) > 0 else 0.0

In [ ]:
"""
Multilingual semantic embedding generation
"""
from sentence_transformers import SentenceTransformer
import numpy as np
import os

class EmbeddingGenerator:
    """Generate multilingual semantic embeddings"""

    def __init__(self, model_name: str):
        """
        Args:
            model_name: Name/path of sentence transformer model
        """
        print(f"Loading embedding model: {model_name}")

        if os.path.exists(model_name):
            # Load from local path
            self.model = SentenceTransformer(model_name)
            print(f"✅ Loaded fine-tuned model from {model_name}")
        else:
            # Load from HuggingFace
            self.model = SentenceTransformer(model_name)
            print(f"✅ Loaded pre-trained model")

    def encode(self, text: str, normalize: bool = True) -> np.ndarray:
        """
        Generate embedding for single text

        Args:
            text: Input text
            normalize: Whether to L2-normalize the embedding

        Returns:
            Embedding vector
        """
        embedding = self.model.encode(
            text,
            convert_to_numpy=True,
            normalize_embeddings=normalize
        )
        return embedding

    def encode_batch(self, texts: list, batch_size: int = 32,
                     normalize: bool = True, show_progress: bool = False) -> np.ndarray:
        """
        Generate embeddings for multiple texts

        Args:
            texts: List of texts
            batch_size: Batch size for encoding
            normalize: Whether to L2-normalize embeddings
            show_progress: Show progress bar

        Returns:
            Array of embeddings (n_texts, embedding_dim)
        """
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=normalize,
            show_progress_bar=show_progress
        )
        return embeddings

    def get_embedding_dim(self) -> int:
        """Get embedding dimension"""
        return self.model.get_sentence_embedding_dimension()

In [ ]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.7 MB/s eta 0:00:00


In [ ]:
"""
Hybrid multilingual evidence retrieval
"""
import faiss
import numpy as np
import pickle
from typing import List, Dict
import os

class EvidenceRetriever:
    """Retrieve relevant evidence using FAISS index"""

    def __init__(self, index_path: str, documents_path: str, embedding_generator):
        """
        Args:
            index_path: Path to FAISS index
            documents_path: Path to documents pickle file
            embedding_generator: EmbeddingGenerator instance
        """
        self.embedder = embedding_generator

        # Load index
        if os.path.exists(index_path):
            self.index = faiss.read_index(index_path)
            print(f"✅ Loaded FAISS index from {index_path}")
        else:
            raise FileNotFoundError(f"Index not found: {index_path}")

        # Load documents
        if os.path.exists(documents_path):
            with open(documents_path, 'rb') as f:
                self.documents = pickle.load(f)
            print(f"✅ Loaded {len(self.documents)} documents")
        else:
            raise FileNotFoundError(f"Documents not found: {documents_path}")

    def retrieve(self, claim: str, lang: str = None, top_k: int = 10) -> List[Dict]:
        """
        Retrieve relevant evidence documents

        Args:
            claim: Claim text to verify
            lang: Language of claim (optional)
            top_k: Number of results to retrieve

        Returns:
            List of evidence dictionaries with scores
        """
        # Encode claim
        query_embedding = self.embedder.encode(claim, normalize=True)
        query_embedding = query_embedding.reshape(1, -1).astype('float32')

        # Search index
        distances, indices = self.index.search(query_embedding, top_k)

        # Prepare results
        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx < len(self.documents):
                doc = self.documents[idx]
                results.append({
                    'title': doc.get('title', 'Untitled'),
                    'text': doc.get('text', ''),
                    'language': doc.get('language', 'unknown'),
                    'similarity': float(dist)
                })

        return results

    def retrieve_by_language(self, claim: str, target_lang: str, top_k: int = 10) -> List[Dict]:
        """
        Retrieve evidence prioritizing specific language

        Args:
            claim: Claim text
            target_lang: Preferred language
            top_k: Number of results

        Returns:
            List of evidence dictionaries
        """
        # Retrieve more than needed
        candidates = self.retrieve(claim, top_k=top_k * 3)

        # Filter by language
        same_lang = [doc for doc in candidates if doc['language'] == target_lang]
        other_lang = [doc for doc in candidates if doc['language'] != target_lang]

        # Combine: prioritize same language
        results = same_lang[:top_k]
        if len(results) < top_k:
            results.extend(other_lang[:top_k - len(results)])

        return results[:top_k]

In [ ]:
"""
Quantum-inspired fidelity scoring
"""
import numpy as np

class QuantumFidelityScorer:
    """Compute quantum-inspired semantic fidelity"""

    def __init__(self):
        pass

    def compute_fidelity(self, claim_embedding: np.ndarray,
                        evidence_embedding: np.ndarray) -> float:
        """
        Compute F = cos²(θ) between claim and evidence

        Args:
            claim_embedding: Claim embedding vector
            evidence_embedding: Evidence embedding vector

        Returns:
            Fidelity score (0 to 1)
        """
        # Normalize vectors
        u = claim_embedding / (np.linalg.norm(claim_embedding) + 1e-10)
        v = evidence_embedding / (np.linalg.norm(evidence_embedding) + 1e-10)

        # Cosine similarity
        cos_theta = np.dot(u, v)

        # Clip to valid range [-1, 1]
        cos_theta = np.clip(cos_theta, -1, 1)

        # Fidelity (squared cosine)
        fidelity = (cos_theta + 1) / 2

        return float(fidelity)

    def compute_fidelity_batch(self, claim_embedding: np.ndarray,
                               evidence_embeddings: np.ndarray) -> np.ndarray:
        """
        Compute fidelity for multiple evidence pieces

        Args:
            claim_embedding: Single claim embedding
            evidence_embeddings: Array of evidence embeddings (n_evidence, dim)

        Returns:
            Array of fidelity scores
        """
        fidelities = []
        for evidence_emb in evidence_embeddings:
            f = self.compute_fidelity(claim_embedding, evidence_emb)
            fidelities.append(f)
        return np.array(fidelities)

    def average_fidelity(self, claim_embedding: np.ndarray,
                        evidence_embeddings: np.ndarray) -> float:
        """
        Compute average fidelity across all evidence

        Args:
            claim_embedding: Claim embedding
            evidence_embeddings: Evidence embeddings

        Returns:
            Average fidelity score
        """
        fidelities = self.compute_fidelity_batch(claim_embedding, evidence_embeddings)
        return float(np.mean(fidelities)) if len(fidelities) > 0 else 0.0

In [ ]:
"""
Evidence support ratio calculation
"""
import numpy as np
from typing import Dict

class SupportRatioCalculator:
    """Calculate evidence support ratio β"""

    def __init__(self, support_threshold: float = 0.85):
        """
        Args:
            support_threshold: Threshold for considering evidence as supporting
        """
        self.support_threshold = support_threshold

    def compute_ratio(self, fidelities: np.ndarray) -> float:
        """
        Compute β = (# supporting evidence) / total

        Args:
            fidelities: Array of fidelity scores

        Returns:
            Support ratio β (0 to 1)
        """
        if len(fidelities) == 0:
            return 0.0

        supporting = np.sum(fidelities >= self.support_threshold)
        total = len(fidelities)

        beta = supporting / total

        return float(beta)

    def get_support_details(self, fidelities: np.ndarray) -> Dict:
        """
        Return detailed breakdown of evidence support

        Args:
            fidelities: Array of fidelity scores

        Returns:
            Dictionary with support statistics
        """
        if len(fidelities) == 0:
            return {
                'supporting': 0,
                'contradicting': 0,
                'neutral': 0,
                'total': 0,
                'ratio': 0.0
            }

        supporting = np.sum(fidelities >= self.support_threshold)
        contradicting = np.sum(fidelities < 0.70)
        neutral = np.sum((fidelities >= 0.70) & (fidelities < self.support_threshold))

        return {
            'supporting': int(supporting),
            'contradicting': int(contradicting),
            'neutral': int(neutral),
            'total': len(fidelities),
            'ratio': float(supporting / len(fidelities))
        }

    def classify_evidence(self, fidelity: float) -> str:
        """
        Classify single evidence as supporting/neutral/contradicting

        Args:
            fidelity: Fidelity score

        Returns:
            Classification string
        """
        if fidelity >= self.support_threshold:
            return 'SUPPORTING'
        elif fidelity < 0.3:
            return 'CONTRADICTING'
        else:
            return 'NEUTRAL'

In [ ]:
"""
Evidence confidence gate - decides if claim has enough support to answer
"""
from typing import Dict

class EvidenceGate:
    """
    Controls whether the system should answer based on three signals:
    - nli_score:   Does evidence actually entail the claim? (NLI score)
    - similarity:  Are retrieved docs semantically close? (fidelity)
    - support_ratio: What fraction of evidence supports the claim? (beta)
    """

    def __init__(self, nli_min: float = 0.55, similarity_min: float = 0.80, support_min: float = 0.40):
        self.nli_min = nli_min
        self.similarity_min = similarity_min
        self.support_min = support_min

    def should_answer(self, nli_score: float, similarity: float, support_ratio: float) -> bool:
        return (
            nli_score >= self.nli_min and
            similarity >= self.similarity_min and
            support_ratio >= self.support_min
        )

    def get_verdict(self, nli_score: float, similarity: float, support_ratio: float) -> Dict:
        nli_ok = nli_score >= self.nli_min
        sim_ok = similarity >= self.similarity_min
        sup_ok = support_ratio >= self.support_min

        gates_passed = sum([nli_ok, sim_ok, sup_ok])

        # Meaningful verdict based on which gates failed
        if gates_passed == 3:
            verdict = "SUPPORTED"
        elif gates_passed == 0:
            verdict = "REFUTED"
        elif not nli_ok:
            verdict = "INSUFFICIENT_EVIDENCE"   # docs retrieved but don't entail claim
        elif not sup_ok:
            verdict = "CONFLICTING_EVIDENCE"    # some support but not enough
        else:
            verdict = "UNVERIFIABLE"            # borderline case

        return {
            'verdict': verdict,
            'should_answer': gates_passed == 3,
            'gates_passed': gates_passed,
            'details': {
                'nli_score':     {'value': nli_score,     'threshold': self.nli_min,        'passed': nli_ok},
                'similarity':    {'value': similarity,    'threshold': self.similarity_min,  'passed': sim_ok},
                'support_ratio': {'value': support_ratio, 'threshold': self.support_min,     'passed': sup_ok},
            }
        }

    def update_thresholds(self, nli_min=None, similarity_min=None, support_min=None):
        if nli_min is not None:
            self.nli_min = nli_min
        if similarity_min is not None:
            self.similarity_min = similarity_min
        if support_min is not None:
            self.support_min = support_min

In [ ]:
"""
Build FAISS index from FineWiki data
"""
import sys
import os
try:
  BASE_PATH = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
  BASE_PATH = os.getcwd()

sys.path.append(BASE_PATH)

import faiss
import numpy as np
import pickle
from tqdm import tqdm

def build_evidence_index():
    """Build FAISS index from FineWiki articles"""

    print("=" * 60)
    print("BUILDING EVIDENCE INDEX")
    print("=" * 60)

    # Load FineWiki data
    print("\nLoading FineWiki data...")
    # Note: load_data and Config are assumed to be defined globally as per previous cells

    # Initialize embedder
    print("\nInitializing embedding model...")
    if os.path.exists(Config.FINETUNED_MODEL_PATH):
        embedder = EmbeddingGenerator(Config.FINETUNED_MODEL_PATH)
    else:
        embedder = EmbeddingGenerator(Config.EMBEDDING_MODEL)

    # Prepare documents
    print("\nPreparing documents...")
    all_documents = []
    all_texts = []

    for lang in Config.SUPPORTED_LANGUAGES:
        # Fix: Assign the result to 'articles' and fix indentation
        articles = download_finewiki(lang, Config.FINEWIKI_SUBSET_SIZE)

        for article in articles:
            # Truncate text for embedding
            text = truncate_text(article['text'], max_chars=500)

            all_documents.append({
                'title': article['title'],
                'text': text,
                'language': lang
            })
            all_texts.append(text)

    print(f"\nTotal documents: {len(all_documents)}")

    # Generate embeddings
    print("\nGenerating embeddings...")
    embeddings = embedder.encode_batch(
        all_texts,
        batch_size=32,
        normalize=True,
        show_progress=True
    )

    # Create FAISS index
    print("\nCreating FAISS index...")
    dimension = embeddings.shape[1]
    print(f"Embedding dimension: {dimension}")

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings.astype('float32'))

    print(f"Index size: {index.ntotal} vectors")

    # Save index
    print(f"\nSaving index to {Config.EVIDENCE_INDEX_PATH}...")
    faiss.write_index(index, Config.EVIDENCE_INDEX_PATH)

    # Save documents
    print(f"Saving documents to {Config.DOCUMENTS_PATH}...")
    with open(Config.DOCUMENTS_PATH, 'wb') as f:
        pickle.dump(all_documents, f)

    print("\n" + "=" * 60)
    print("✅ EVIDENCE INDEX BUILT SUCCESSFULLY")
    print("=" * 60)

if __name__ == "__main__":
    build_evidence_index()

BUILDING EVIDENCE INDEX

Loading FineWiki data...

Initializing embedding model...
Loading embedding model: sentence-transformers/LaBSE


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

✅ 48209 articles for ar



Processing ta: 100%|██████████| 50000/50000 [00:31<00:00, 1578.79it/s]


✅ 49936 articles for ta

Total documents: 98145

Generating embeddings...


Batches:   0%|          | 0/3068 [00:00<?, ?it/s]


Creating FAISS index...
Embedding dimension: 768
Index size: 98145 vectors

Saving index to /content/data/indices/evidence_index.faiss...
Saving documents to /content/data/indices/documents.pkl...

✅ EVIDENCE INDEX BUILT SUCCESSFULLY


training

In [ ]:
"""
Stage 1: Fine-tune embeddings using FineWeb2 data
"""
import sys
import os
try:
    BASE_PATH = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    BASE_PATH = os.getcwd()

sys.path.append(BASE_PATH)

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from tqdm import tqdm
import random

def create_training_pairs_balanced(documents, num_pairs=100000, positive_ratio=0.5):
    """
    Create balanced positive and negative training pairs

    Args:
        documents: List of document dictionaries
        num_pairs: Total number of pairs to create
        positive_ratio: Ratio of positive pairs (0.5 = 50/50 split)

    Returns:
        List of InputExample objects
    """
    import random

    num_positive = int(num_pairs * positive_ratio)
    num_negative = num_pairs - num_positive

    print(f"\nCreating {num_pairs} training pairs:")
    print(f"  Positive pairs: {num_positive}")
    print(f"  Negative pairs: {num_negative}")

    examples = []
    random.shuffle(documents)

    # ============================================
    # CREATE POSITIVE PAIRS
    # ============================================
    print("\n1. Creating positive pairs (similar texts)...")
    positive_count = 0

    for doc in tqdm(documents, desc="Positive pairs"):
        if positive_count >= num_positive:
            break

        text = doc.get('text', '')
        if len(text) < 200:
            continue

        # Split into chunks
        chunks = split_into_chunks(text, chunk_size=200, overlap=50)

        # Create pairs from same document (semantically similar)
        for i in range(len(chunks) - 1):
            if positive_count >= num_positive:
                break

            examples.append(InputExample(
                texts=[chunks[i], chunks[i + 1]],
                label=1.0  # Similar
            ))
            positive_count += 1

    print(f"✅ Created {positive_count} positive pairs")

    # ============================================
    # CREATE NEGATIVE PAIRS
    # ============================================
    print("\n2. Creating negative pairs (dissimilar texts)...")
    negative_count = 0

    # Filter documents by length for efficiency
    valid_docs = [d for d in documents if len(d.get('text', '')) >= 100]

    pbar = tqdm(total=num_negative, desc="Negative pairs")

    while negative_count < num_negative:
        # Pick two random documents
        doc1 = random.choice(valid_docs)
        doc2 = random.choice(valid_docs)

        # Ensure different documents
        text1 = doc1.get('text', '')
        text2 = doc2.get('text', '')

        if text1 == text2:
            continue

        # Take first 200 chars from each
        text1 = text1[:200].strip()
        text2 = text2[:200].strip()

        # Filter very short texts
        if len(text1) < 100 or len(text2) < 100:
            continue

        examples.append(InputExample(
            texts=[text1, text2],
            label=0.0  # Dissimilar
        ))
        negative_count += 1
        pbar.update(1)

    pbar.close()
    print(f"✅ Created {negative_count} negative pairs")

    # ============================================
    # SHUFFLE AND RETURN
    # ============================================
    random.shuffle(examples)

    # Verify counts
    pos_count = sum(1 for e in examples if e.label > 0.5)
    neg_count = sum(1 for e in examples if e.label < 0.5)

    print(f"\n✅ Total pairs created: {len(examples)}")
    print(f"   Positive: {pos_count} ({pos_count/len(examples)*100:.1f}%)")
    print(f"   Negative: {neg_count} ({neg_count/len(examples)*100:.1f}%)")

    return examples

def finetune_embeddings():
    """Fine-tune embedding model on FineWeb2 data with validation"""

    print("=" * 60)
    print("STAGE 1: FINE-TUNING EMBEDDINGS")
    print("=" * 60)

    Config.display()

    # ============================================
    # LOAD DATA
    # ============================================
    print("\nLoading FineWeb2 data...")
    fineweb2_data = load_data('fineweb2_data.pkl')

    # Combine all documents
    all_documents = []
    for lang in Config.SUPPORTED_LANGUAGES:
        docs = fineweb2_data.get(lang, [])
        all_documents.extend(docs)
        print(f"  {lang}: {len(docs)} documents")

    print(f"Total documents: {len(all_documents)}")

    # ============================================
    # LOAD BASE MODEL
    # ============================================
    print(f"\nLoading base model: {Config.EMBEDDING_MODEL}")
    model = SentenceTransformer(Config.EMBEDDING_MODEL)
    print(f"✅ Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

    # ============================================
    # CREATE TRAINING PAIRS
    # ============================================
    training_pairs = create_training_pairs_balanced(
        all_documents,
        num_pairs=Config.STAGE1_NUM_PAIRS,
        positive_ratio=Config.STAGE1_POSITIVE_RATIO
    )

    # ============================================
    # SPLIT TRAIN/VALIDATION
    # ============================================
    print(f"\nSplitting data (train/val: {1-Config.STAGE1_VALIDATION_SPLIT:.0%}/{Config.STAGE1_VALIDATION_SPLIT:.0%})...")

    split_idx = int(len(training_pairs) * (1 - Config.STAGE1_VALIDATION_SPLIT))
    train_pairs = training_pairs[:split_idx]
    val_pairs = training_pairs[split_idx:]

    print(f"  Train pairs: {len(train_pairs)}")
    print(f"  Validation pairs: {len(val_pairs)}")

    # ============================================
    # PREPARE DATALOADERS
    # ============================================
    print("\nPreparing dataloaders...")
    train_dataloader = DataLoader(
        train_pairs,
        shuffle=True,
        batch_size=Config.STAGE1_BATCH_SIZE
    )

    # ============================================
    # DEFINE LOSS FUNCTION
    # ============================================
    print(f"\nSetting up loss function: {Config.STAGE1_LOSS_TYPE}")

    if Config.STAGE1_LOSS_TYPE == 'contrastive':
        train_loss = losses.ContrastiveLoss(model)
        print("✅ Using ContrastiveLoss (better for positive/negative pairs)")
    else:
        train_loss = losses.CosineSimilarityLoss(model)
        print("✅ Using CosineSimilarityLoss")

    # ============================================
    # SETUP EVALUATOR
    # ============================================
    print("\nSetting up validation evaluator...")
    from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

    # Create evaluation data (use subset for speed)
    eval_subset_size = min(500, len(val_pairs))
    sentences1 = [pair.texts[0] for pair in val_pairs[:eval_subset_size]]
    sentences2 = [pair.texts[1] for pair in val_pairs[:eval_subset_size]]
    scores = [pair.label for pair in val_pairs[:eval_subset_size]]

    evaluator = EmbeddingSimilarityEvaluator(
        sentences1,
        sentences2,
        scores,
        name='validation',
        show_progress_bar=True
    )

    print(f"✅ Evaluator ready with {eval_subset_size} validation pairs")

    # ============================================
    # TRAINING
    # ============================================
    print("\n" + "=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    print(f"Epochs: {Config.STAGE1_EPOCHS}")
    print(f"Batch size: {Config.STAGE1_BATCH_SIZE}")
    print(f"Learning rate: {Config.STAGE1_LEARNING_RATE}")
    print(f"Warmup steps: {Config.STAGE1_WARMUP_STEPS}")
    print(f"Evaluation every: {Config.STAGE1_EVAL_STEPS} steps")
    print(f"Total training steps: {len(train_dataloader) * Config.STAGE1_EPOCHS}")
    print("=" * 60)

    # Train the model
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=Config.STAGE1_EPOCHS,
        warmup_steps=Config.STAGE1_WARMUP_STEPS,
        evaluator=evaluator,
        evaluation_steps=Config.STAGE1_EVAL_STEPS,
        output_path=Config.FINETUNED_MODEL_PATH,
        save_best_model=Config.STAGE1_SAVE_BEST,
        checkpoint_path='checkpoints/stage1',  # ✅ saves every epoch
    checkpoint_save_steps=5625,
        show_progress_bar=True,
        optimizer_params={'lr': Config.STAGE1_LEARNING_RATE}
    )

    # ============================================
    # POST-TRAINING VALIDATION
    # ============================================
    print("\n" + "=" * 60)
    print("TRAINING COMPLETE - FINAL VALIDATION")
    print("=" * 60)

    # Load best model
    if Config.STAGE1_SAVE_BEST and os.path.exists(Config.FINETUNED_MODEL_PATH):
        print(f"Loading best model from: {Config.FINETUNED_MODEL_PATH}")
        model = SentenceTransformer(Config.FINETUNED_MODEL_PATH)

    # Final evaluation
    print("\nRunning final validation...")
    final_score = evaluator(model)
    print(f"Final validation score: {final_score}")

    # Test cross-lingual similarity
    print("\n" + "=" * 60)
    print("TESTING CROSS-LINGUAL EMBEDDINGS")
    print("=" * 60)

    # Test with sample texts
    test_samples = {
        'ar': "هذا كتاب جيد",  # "This is a good book" in Arabic
        'ta': "இது ஒரு நல்ல புத்தகம்",  # "This is a good book" in Tamil
        'en': "This is a good book"
    }

    embeddings = {}
    for lang, text in test_samples.items():
        embeddings[lang] = model.encode(text)
        print(f"{lang}: {text[:50]}...")

    # Compute similarities
    import numpy as np
    print("\nCross-lingual similarities:")
    print(f"  Arabic-Tamil: {np.dot(embeddings['ar'], embeddings['ta']):.3f}")
    print(f"  Arabic-English: {np.dot(embeddings['ar'], embeddings['en']):.3f}")
    print(f"  Tamil-English: {np.dot(embeddings['ta'], embeddings['en']):.3f}")
    print("\n(Should be > 0.6 for good cross-lingual alignment)")

    # ============================================
    # SAVE FINAL MODEL
    # ============================================
    print("\n" + "=" * 60)
    print("✅ STAGE 1 COMPLETE")
    print("=" * 60)
    print(f"Fine-tuned model saved to: {Config.FINETUNED_MODEL_PATH}")
    print(f"Model is ready for Stage 2 (threshold optimization)")
    print("=" * 60)

if __name__ == "__main__":
    finetune_embeddings()




STAGE 1: FINE-TUNING EMBEDDINGS
CONFIGURATION
Languages       : ar, ta
FineWiki subset : 50000 articles/lang
FineWeb2 subset : 80000 docs/lang
Training pairs  : 100000 (pos/neg: 50%/50%)
Embedding model : sentence-transformers/LaBSE
NLI model       : MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Epochs          : 1
Batch size      : 16
Thresholds      : nli=0.55, similarity=0.8, support=0.4

Loading FineWeb2 data...
  ar: 80000 documents
  ta: 80000 documents
Total documents: 160000

Loading base model: sentence-transformers/LaBSE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded. Embedding dimension: 768

Creating 100000 training pairs:
  Positive pairs: 50000
  Negative pairs: 50000

1. Creating positive pairs (similar texts)...


Positive pairs:   1%|▏         | 2308/160000 [00:00<00:38, 4143.94it/s]


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Running final validation...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

TypeError: unsupported format string passed to dict.__format__

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load your fine-tuned model
model = SentenceTransformer('/content/models/embeddings_finetuned')

# Test 1 — cross-lingual similarity (Arabic-Tamil same meaning)
test_pairs = [
    # Same meaning, different language → should be HIGH
    ("هذا كتاب جيد", "இது ஒரு நல்ல புத்தகம்"),        # "This is a good book"
    ("الطقس جميل اليوم", "இன்று வானிலை அழகாக உள்ளது"), # "Weather is nice today"

    # Different meaning → should be LOW
    ("هذا كتاب جيد", "இன்று வானிலை அழகாக உள்ளது"),
    ("الطقس جميل اليوم", "இது ஒரு நல்ல புத்தகம்"),
]

print("Cross-lingual similarity test:")
print("-" * 50)
for t1, t2 in test_pairs:
    e1 = model.encode(t1, normalize_embeddings=True)
    e2 = model.encode(t2, normalize_embeddings=True)
    sim = float(np.dot(e1, e2))
    print(f"Similarity: {sim:.3f} | {t1[:20]} ↔ {t2[:20]}")

# Test 2 — fact checking claim vs evidence
print("\nFact-checking relevance test:")
print("-" * 50)
claim = "كورونا فيروس خطير"  # "Corona virus is dangerous"
evidence_good = "فيروس كورونا المستجد يسبب مرضاً خطيراً"  # "Covid causes serious illness"
evidence_bad  = "الطقس في باريس جميل جداً"  # "Weather in Paris is nice"

ec = model.encode(claim, normalize_embeddings=True)
eg = model.encode(evidence_good, normalize_embeddings=True)
eb = model.encode(evidence_bad, normalize_embeddings=True)

print(f"Claim vs relevant evidence : {float(np.dot(ec, eg)):.3f}  ← should be HIGH")
print(f"Claim vs irrelevant evidence: {float(np.dot(ec, eb)):.3f}  ← should be LOW")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cross-lingual similarity test:
--------------------------------------------------
Similarity: 0.900 | هذا كتاب جيد ↔ இது ஒரு நல்ல புத்தகம
Similarity: 0.856 | الطقس جميل اليوم ↔ இன்று வானிலை அழகாக உ
Similarity: 0.648 | هذا كتاب جيد ↔ இன்று வானிலை அழகாக உ
Similarity: 0.588 | الطقس جميل اليوم ↔ இது ஒரு நல்ல புத்தகம

Fact-checking relevance test:
--------------------------------------------------
Claim vs relevant evidence : 0.891  ← should be HIGH
Claim vs irrelevant evidence: 0.638  ← should be LOW


In [ ]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ============================================
# LOAD EVERYTHING
# ============================================
print("Loading models and index...")

model = SentenceTransformer('/content/models/embeddings_finetuned')
index = faiss.read_index('data/indices/evidence_index.faiss')

with open('data/indices/documents.pkl', 'rb') as f:
    documents = pickle.load(f)

nli = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=-1
)

print(f"✅ Ready! Index has {index.ntotal} documents\n")

# ============================================
# DEMO FUNCTION
# ============================================
def verify_claim(claim: str, top_k: int = 5):
    print("=" * 60)
    print(f"CLAIM: {claim}")
    print("=" * 60)

    # Step 1 — Retrieve evidence
    query_emb = model.encode(claim, normalize_embeddings=True)
    query_emb = query_emb.reshape(1, -1).astype('float32')
    distances, indices = index.search(query_emb, top_k)

    # Step 2 — NLI check each evidence
    print(f"\nTOP {top_k} EVIDENCE PIECES:")
    print("-" * 60)

    nli_scores = []
    similarities = []

    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        if idx >= len(documents):
            continue

        doc = documents[idx]
        text = doc.get('text', '')[:300]
        lang = doc.get('language', '?')

        # NLI score
        result = nli(
            text[:512],
            candidate_labels=["supported", "refuted"],
            hypothesis_template="The following claim is {}: " + claim
        )
        supported_idx = result['labels'].index('supported')
        nli_score = result['scores'][supported_idx]
        nli_scores.append(nli_score)
        similarities.append(float(dist))

        verdict = "✅ SUPPORTING" if nli_score >= 0.55 else "❌ REFUTING" if nli_score < 0.35 else "⚠️ NEUTRAL"

        print(f"\n[{rank+1}] Similarity: {dist:.3f} | NLI: {nli_score:.3f} | {verdict}")
        print(f"    Lang: {lang}")
        print(f"    Text: {text[:120]}...")

    # Step 3 — EvidenceGate decision
    avg_nli        = float(np.mean(nli_scores))
    avg_similarity = float(np.mean(similarities))
    support_ratio  = sum(1 for s in nli_scores if s >= 0.55) / len(nli_scores)

    print("\n" + "=" * 60)
    print("VERDICT")
    print("=" * 60)
    print(f"NLI Score      : {avg_nli:.3f}  (threshold: 0.55)")
    print(f"Avg Similarity : {avg_similarity:.3f}  (threshold: 0.80)")
    print(f"Support Ratio  : {support_ratio:.3f}  (threshold: 0.40)")

    if avg_nli >= 0.55 and avg_similarity >= 0.80 and support_ratio >= 0.40:
        print("\n🟢 FINAL VERDICT: SUPPORTED")
    elif avg_nli < 0.35:
        print("\n🔴 FINAL VERDICT: REFUTED")
    elif support_ratio < 0.40:
        print("\n🟡 FINAL VERDICT: CONFLICTING EVIDENCE")
    else:
        print("\n⚪ FINAL VERDICT: INSUFFICIENT EVIDENCE")

# ============================================
# TEST IT
# ============================================
verify_claim("فيروس كورونا خطير على الصحة")  # "Corona virus is dangerous to health"

Loading models and index...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

✅ Ready! Index has 98145 documents

CLAIM: فيروس كورونا خطير على الصحة

TOP 5 EVIDENCE PIECES:
------------------------------------------------------------

[1] Similarity: 0.478 | NLI: 0.611 | ✅ SUPPORTING
    Lang: ar
    Text: # اكتشاف العوامل الممرضة المسببة للمرض
اكتشاف العوامل الممرضة المسببة للمرض (بالإنجليزية: Discovery of disease-causing p...

[2] Similarity: 0.477 | NLI: 0.550 | ⚠️ NEUTRAL
    Lang: ar
    Text: # التطعيم ضد فيروس كورونا في إسرائيل
بدأ برنامج التلقيح ضد كوفيد-19 في إسرائيل، والذي سمي رسميًا باسم «أعطِ كتفًا» (بالع...

[3] Similarity: 0.464 | NLI: 0.588 | ✅ SUPPORTING
    Lang: ta
    Text: # சார்சு
தீவிர கடிய மூச்சியக்க கூட்டறிகுறி (Severe acute respiratory syndrome என்பதன் ஆங்கிலச் சுருக்கம் சார்சு) அல்லது ...

[4] Similarity: 0.464 | NLI: 0.570 | ✅ SUPPORTING
    Lang: ar
    Text: # التطعيم ضد فيروس كورونا في الفلبين
لقاح كوفيد-19 في الفلبين حملة تمنيع جارية ضد فيروس كورونا المستجد المسبب المتلازمة ...

[5] Similarity: 0.459 | NLI: 0.445 | ⚠️ NEUTRAL
    L

In [ ]:
# Step 1 — check which model built the index
import os
print(os.path.exists('/content/models/embeddings_finetuned'))

True


In [ ]:
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("Loading fine-tuned model...")
model = SentenceTransformer('/content/models/embeddings_finetuned')

print("Loading documents...")
with open('data/cache/finewiki_data.pkl', 'rb') as f:
    finewiki_data = pickle.load(f)

# Prepare all documents
all_documents = []
all_texts = []

for lang in ['ar', 'ta']:
    articles = finewiki_data[lang]
    print(f"  {lang}: {len(articles)} articles")
    for article in articles:
        text = article.get('text', '')[:500]
        if len(text) > 100:
            all_documents.append({
                'title': article.get('title', ''),
                'text': text,
                'language': lang
            })
            all_texts.append(text)

print(f"\nTotal documents: {len(all_documents)}")

# Generate embeddings in batches
print("\nGenerating embeddings (this takes ~20 mins)...")
embeddings = model.encode(
    all_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Build FAISS index
print("\nBuilding FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype('float32'))
print(f"Index size: {index.ntotal} vectors")

# Save
os.makedirs('data/indices', exist_ok=True)
faiss.write_index(index, 'data/indices/evidence_index.faiss')
with open('data/indices/documents.pkl', 'wb') as f:
    pickle.dump(all_documents, f)

print("✅ Index rebuilt with fine-tuned model!")

Loading fine-tuned model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading documents...
  ar: 48209 articles
  ta: 49936 articles

Total documents: 98145

Generating embeddings (this takes ~20 mins)...


Batches:   0%|          | 0/1534 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
os.makedirs('/content/drive/MyDrive/factcheck_models', exist_ok=True)
shutil.copytree(
    '/content/models/embeddings_finetuned',
    '/content/drive/MyDrive/factcheck_models/embeddings_finetuned'
)
print("✅ Backed up!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Backed up!


train_baseline.py

In [ ]:
import json
import torch
import argparse
import logging
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import evaluate
import jsonlines
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NEI": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

MODEL_NAMES = {
    "mbert":      "bert-base-multilingual-cased",
    "xlmr":       "xlm-roberta-base",
    "xlmr-large": "xlm-roberta-large",
    "muril":      "google/muril-base-cased",
    "arabert":    "aubmindlab/bert-base-arabertv2",
}

class FactCheckDataset(Dataset):
    def __init__(self, items, tokenizer, max_length=256):
        self.items = items
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        encoding = self.tokenizer(
            item["claim"],
            item.get("evidence", ""),
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        label = LABEL2ID.get(item["label"], 2)
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label, dtype=torch.long),
        }

def load_split(data_dir, split):
    path = f"{data_dir}/{split}.jsonl"
    with jsonlines.open(path) as reader:
        return list(reader)

def compute_metrics(eval_pred):
    metric_acc = evaluate.load("accuracy")
    metric_f1  = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1  = metric_f1.compute(predictions=predictions, references=labels, average="macro", labels=[0,1,2])
    f1_per = metric_f1.compute(predictions=predictions, references=labels, average=None, labels=[0,1,2])
    result = {"accuracy": acc["accuracy"], "macro_f1": f1["f1"]}
    for i, name in ID2LABEL.items():
        result[f"f1_{name.lower()}"] = f1_per["f1"][i]
    return result

def train(model_key, data_dir, output_dir, epochs=5, batch_size=16):
    model_name = MODEL_NAMES.get(model_key, model_key)
    logger.info(f"Training: {model_name}")
    logger.info(f"Epochs: {epochs}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
    )

    train_dataset = FactCheckDataset(load_split(data_dir, "train"), tokenizer)
    val_dataset   = FactCheckDataset(load_split(data_dir, "val"),   tokenizer)
    test_dataset  = FactCheckDataset(load_split(data_dir, "test"),  tokenizer)

    logger.info(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        warmup_steps=200,
        weight_decay=0.01,
        learning_rate=2e-5,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_results = trainer.evaluate(test_dataset)
    logger.info("\\n=== TEST RESULTS ===")
    for k, v in test_results.items():
        logger.info(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/test_results.json", "w") as f:
        json.dump(test_results, f, indent=2)

    trainer.save_model(f"{output_dir}/best_model")
    tokenizer.save_pretrained(f"{output_dir}/best_model")
    logger.info(f"Saved to {output_dir}/best_model")
    return test_results

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model",      default="xlmr", choices=list(MODEL_NAMES.keys()))
    parser.add_argument("--data_dir",   default="data/final/")
    parser.add_argument("--output_dir", default="checkpoints/baseline/")
    parser.add_argument("--epochs",     type=int, default=2)
    parser.add_argument("--batch_size", type=int, default=16)
    args = parser.parse_args()
    os.makedirs(args.output_dir, exist_ok=True)
    train(args.model, args.data_dir, args.output_dir, args.epochs, args.batch_size)


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("models/embeddings_finetuned")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# ============================================
# FULL EMBEDDING MODEL EVALUATION SCRIPT
# ============================================

import numpy as np
from sentence_transformers import SentenceTransformer, InputExample
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import random

# --------------------------------------------
# 1. LOAD TRAINED MODEL
# --------------------------------------------
model_path = "models/embeddings_finetuned"   # your folder
model = SentenceTransformer(model_path)
print("✅ Model loaded")


# --------------------------------------------
# 2. RELOAD YOUR DATA
# --------------------------------------------
fineweb2_data = load_data('fineweb2_data.pkl')   # same as training

all_documents = []
for lang in Config.SUPPORTED_LANGUAGES:
    all_documents.extend(fineweb2_data.get(lang, []))

print("Total documents:", len(all_documents))


# --------------------------------------------
# 3. RECREATE PAIRS (same logic as training)
# --------------------------------------------
pairs = create_training_pairs_balanced(
    all_documents,
    num_pairs=Config.STAGE1_NUM_PAIRS,
    positive_ratio=Config.STAGE1_POSITIVE_RATIO
)

# split exactly same way
split_idx = int(len(pairs) * (1 - Config.STAGE1_VALIDATION_SPLIT))
val_pairs = pairs[split_idx:]

print("Validation pairs:", len(val_pairs))


# --------------------------------------------
# 4. ENCODE ALL SENTENCES (FAST batch mode)
# --------------------------------------------
s1 = [p.texts[0] for p in val_pairs]
s2 = [p.texts[1] for p in val_pairs]
labels = np.array([int(p.label) for p in val_pairs])

print("Encoding sentences...")

emb1 = model.encode(s1, batch_size=64, show_progress_bar=True)
emb2 = model.encode(s2, batch_size=64, show_progress_bar=True)

# cosine similarity
sims = np.sum(emb1 * emb2, axis=1)


# --------------------------------------------
# 5. CORRELATION METRICS (embedding quality)
# --------------------------------------------
pearson = pearsonr(sims, labels)[0]
spearman = spearmanr(sims, labels)[0]

print("\n📊 Correlation Metrics")
print("---------------------")
print(f"Pearson  : {pearson:.4f}")
print(f"Spearman : {spearman:.4f}")


# --------------------------------------------
# 6. FIND BEST THRESHOLD
# --------------------------------------------
best_acc = 0
best_t = 0

for t in np.arange(0.3, 0.9, 0.01):
    preds = (sims > t).astype(int)
    acc = accuracy_score(labels, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print(f"\nBest threshold: {best_t:.2f}")


# --------------------------------------------
# 7. FINAL CLASSIFICATION METRICS
# --------------------------------------------
preds = (sims > best_t).astype(int)

print("\n📊 Classification Metrics")
print("------------------------")
print(f"Accuracy  : {accuracy_score(labels, preds):.4f}")
print(f"F1 Score  : {f1_score(labels, preds):.4f}")
print(f"Precision : {precision_score(labels, preds):.4f}")
print(f"Recall    : {recall_score(labels, preds):.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model loaded
Total documents: 160000

Creating 100000 training pairs:
  Positive pairs: 50000
  Negative pairs: 50000

1. Creating positive pairs (similar texts)...


Positive pairs:   1%|▏         | 2117/160000 [00:00<00:05, 28591.85it/s]

✅ Created 50000 positive pairs

2. Creating negative pairs (dissimilar texts)...



Negative pairs: 100%|██████████| 50000/50000 [00:01<00:00, 35046.37it/s]


✅ Created 50000 negative pairs

✅ Total pairs created: 100000
   Positive: 50000 (50.0%)
   Negative: 50000 (50.0%)
Validation pairs: 10000
Encoding sentences...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]


📊 Correlation Metrics
---------------------
Pearson  : 0.9505
Spearman : 0.8653

Best threshold: 0.68

📊 Classification Metrics
------------------------
Accuracy  : 0.9925
F1 Score  : 0.9924
Precision : 0.9909
Recall    : 0.9939


STAY OFF THE WEEEEEEEEEEE

Scrape factcrescendo